# Lesson 01 — اولین تماس با LLM

**پیش‌نیاز:** Lesson 00 تمام (GPU + پکیج‌ها نصب)

**هدف:** اولین بار یک مدل واقعی را load کنی، prompt بفرستی، و خروجی را کنترل کنی.

**اتصال به محصول:** دکمهٔ «Rewrite with AI» در ورک‌اسپیس همین جریان است — system prompt + متن بلاک → مدل → متن جدید.

---

## چرا این درس؟
قبل از LoRA، RAG و Browser Agent باید بفهمی:
- مدل چطور **متن** می‌بیند (tokenizer)
- چطور **نقش** system/user را می‌فهمد (chat template)
- چطور **خلاقیت** را کم/زیاد کنی (temperature)
- چطور **طول** خروجی را محدود کنی (max_tokens)

**Runtime → GPU (T4)** — اگر session جدید است، cell نصب Lesson 00 را یک‌بار دوباره اجرا کن.

## 0) چک سریع محیط (اگر session تازه است)

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU off — Runtime → Change runtime type → GPU, then Restart"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# فقط اگر session تازه است — در Lesson 00 قبلاً نصب کردی
%pip install -q "transformers>=4.44" "accelerate" "sentencepiece" "einops"

## 1) بارگذاری مدل

مدل: `Qwen/Qwen2.5-1.5B-Instruct` — سبک، مناسب T4، همان مدلی که در Lesson 00 tokenizer آن را تست کردی.

اولین بار ~۳GB دانلود می‌شود.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded on:", next(model.parameters()).device)

## 2) تابع کمکی `generate` — هستهٔ محصول

این الگو بعداً در `packages/ai-core/client.py` می‌رود. الان در Colab یاد می‌گیری.

In [ ]:
def generate(
    messages: list[dict],
    *,
    temperature: float = 0.2,
    max_new_tokens: int = 256,
    do_sample: bool | None = None,
) -> str:
    """messages: [{"role": "system"|"user"|"assistant", "content": "..."}]"""
    if do_sample is None:
        do_sample = temperature > 0

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if do_sample else None,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 3) Tokenizer و Chat Template — مدل چه می‌بیند؟

مدل مستقیم JSON نمی‌خواند؛ tokenizer متن را به **token** و chat template را به **فرمت مخصوص مدل** تبدیل می‌کند.

In [ ]:
BLOCK = "Ship AI workspace mvp soon. Need pages, RAG, browser agent."

messages = [
    {"role": "system", "content": "You are an AI assistant inside a Notion-like workspace. Be concise."},
    {"role": "user", "content": f"Rewrite this block to be clearer:\n\n{BLOCK}"},
]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
token_ids = tokenizer.encode(prompt_text)

print("--- Prompt (first 600 chars) ---")
print(prompt_text[:600])
print(f"\nToken count (input): {len(token_ids)}")

## 4) اولین generation

In [ ]:
result = generate(messages, temperature=0.2, max_new_tokens=150)
print("--- Model output ---")
print(result)

## 5) Temperature — خلاقیت vs ثبات

| temperature | رفتار |
|-------------|--------|
| `0` یا نزدیک ۰ | تقریباً deterministic — برای rewrite/summarize محصول |
| `0.5–0.8` | کمی تنوع |
| `> 1` | پراکنده‌تر — معمولاً برای محصول workspace بد |

برای **Rewrite with AI** در محصول: `temperature=0.1–0.3`

In [ ]:
for temp in [0.0, 0.3, 0.8]:
    out = generate(messages, temperature=temp, max_new_tokens=80)
    print(f"\n=== temperature={temp} ===")
    print(out)

## 6) System prompt — لحن و قوانین

System prompt = «قوانین دائمی» برای مدل در این مکالمه. در محصول می‌تواند بگوید: فقط متن بلاک برگردان، markdown اضافه نکن، و غیره.

In [ ]:
SYSTEM_PROMPTS = {
    "minimal": "You rewrite workspace blocks. Output only the rewritten text.",
    "formal": "You are a professional editor. Rewrite in formal business English. Output only the block text.",
    "bullets": "Rewrite as 3 short bullet points. Use - for bullets. No intro sentence.",
}

for name, system in SYSTEM_PROMPTS.items():
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Rewrite:\n\n{BLOCK}"},
    ]
    out = generate(msgs, temperature=0.2, max_new_tokens=120)
    print(f"\n=== system: {name} ===")
    print(out)

## 7) max_new_tokens — جلوگیری از پاسخ بی‌پایان

برای summarize کوتاه: `64–128` — برای rewrite بلند: `256`

In [ ]:
summarize_msgs = [
    {"role": "system", "content": "Summarize in at most 2 sentences. No preamble."},
    {"role": "user", "content": BLOCK},
]

for n in [32, 128]:
    out = generate(summarize_msgs, temperature=0.2, max_new_tokens=n)
    print(f"\nmax_new_tokens={n} → {len(out)} chars")
    print(out)

## 8) دادهٔ واقعی ورک‌اسپیس

از نمونهٔ محصول (`data/samples/welcome_page.json`) یک بلاک واقعی:

In [ ]:
REAL_BLOCK = (
    "Build a Notion-like workspace where AI can edit blocks, "
    "answer from workspace knowledge, and run a browser agent that writes results back as pages."
)

msgs = [
    {"role": "system", "content": "You are the AI inside AI Workspace. Rewrite blocks clearly. Output only the new block text."},
    {"role": "user", "content": f"Rewrite this paragraph for a product spec:\n\n{REAL_BLOCK}"},
]
print(generate(msgs, temperature=0.2, max_new_tokens=200))

---

## تمرین‌های Lesson 01 (اجباری)

بلاک زیر را با **۳ prompt مختلف** بازنویسی کن (system و/یا user را عوض کن). بعد در cell بعدی مقایسه کن: کدام برای محصول بهتر است؟

```
Pages and blocks. AI rewrite and summarize. RAG over pages. Browser agent with goto/click/extract. Citations back to sources.
```

**سؤالات مقایسه:**
1. کدام خروجی کوتاه‌تر و قابل‌استفاده‌تر در UI است؟
2. آیا مدل توضیح اضافه («Here is…») گذاشت؟ چطور با system prompt جلوگیری کردی؟
3. temperature=0.2 برای هر سه یکسان بود؟

### چک‌لیست
- [ ] مدل load شد
- [ ] token count ورودی را دیدی
- [ ] تفاوت temperature را دیدی
- [ ] ۳ prompt تمرین را اجرا و مقایسه کردی
- [ ] یک خط در `docs/journal.md` نوشتی

وقتی تمام شد بگو: **درس 01 تمام**

In [ ]:
EXERCISE_BLOCK = (
    "Pages and blocks. AI rewrite and summarize. RAG over pages. "
    "Browser agent with goto/click/extract. Citations back to sources."
)

# --- Prompt 1: خودت system + user را بنویس ---
prompt_1 = [
    {"role": "system", "content": "TODO: your system prompt"},
    {"role": "user", "content": f"TODO: your instruction\n\n{EXERCISE_BLOCK}"},
]

# --- Prompt 2 ---
prompt_2 = [
    {"role": "system", "content": "TODO"},
    {"role": "user", "content": f"TODO\n\n{EXERCISE_BLOCK}"},
]

# --- Prompt 3 ---
prompt_3 = [
    {"role": "system", "content": "TODO"},
    {"role": "user", "content": f"TODO\n\n{EXERCISE_BLOCK}"},
]

for i, p in enumerate([prompt_1, prompt_2, prompt_3], start=1):
    out = generate(p, temperature=0.2, max_new_tokens=200)
    print(f"\n{'='*50}\nPROMPT {i}\n{'='*50}")
    print(out)

## 9) یادداشت مقایسه (بنویس)

In [ ]:
MY_COMPARISON = """
بهترین prompt برای محصول: Prompt #...
چرا: ...
مشکل مدل: ...
"""
print(MY_COMPARISON.strip() or "WARNING: comparison still empty")

## گام بعدی (Lesson 02)

مدل گاهی متن اضافه می‌گذارد. در **Lesson 02** خروجی را **JSON ساختاریافته** می‌کنی:
`{"action": "rewrite", "block_id": "b4", "content": "..."}`

این همان قراردادی است که API ورک‌اسپیس برای ویرایش بلاک استفاده می‌کند.